## Interactive Demonstration With UAV-VisLoc Dataset

### Control keys

-   Esc: Exit
-   A: Previous image
-   W: 10th Previous image
-   D: Next image
-   S: 10th Next image


In [ ]:
from dataclasses import asdict

from pyproj import Transformer
import cv2
import numpy as np
import matplotlib.pyplot as plt
from lightglue import viz2d

from uavcalibration.calibration import Calibration
from uavcalibration.datasets import VisLocDataset
from uavcalibration.transform import *

dataset = VisLocDataset(r"..\datasets\UAV_VisLoc_example")

# 全局变量存储鼠标位置
uav_pos = (0, 0)
satellite_pos = (0, 0)
satellite_lonlat = (0, 0)
data_index = 0


def calibrate():
    calibration = Calibration(uav_data.uav_image)
    calibration.coarse_calibrate(**asdict(uav_data))
    src_shape = uav_data.uav_image.shape
    calibration.transform.adjust_shape(src_shape=(src_shape[1], src_shape[0]))
    calibration.fine_calibrate(
        satellite_image=uav_data.satellite_image,
        satellite_crs=CRSTransform(
            uav_data.satellite_transform, uav_data.satellite_crs
        ),
    )
    return calibration


# 鼠标回调函数
def mouse_callback(event, x, y, flags, param):
    global uav_pos, satellite_pos, satellite_lonlat
    if event == cv2.EVENT_MOUSEMOVE:  # 鼠标移动事件
        uav_pos = (x, y)
        satellite_pos = tuple(
            calibration.transform.apply(np.array(uav_pos, np.float64))
            .ravel()
            .astype(int)
        )
        satellite_lonlat = tuple(
            calibration.transform.crs.apply(np.array(satellite_pos, np.float64)).ravel()
        )
        transformer = Transformer.from_crs(
            calibration.transform.crs.crs, "epsg:4326", always_xy=True
        )
        satellite_lonlat = transformer.transform(*satellite_lonlat)


size = 10

uav_data = dataset[data_index]
calibration = calibrate()

# 创建窗口并绑定回调函数
cv2.namedWindow("UAV Image", cv2.WINDOW_KEEPRATIO)
cv2.namedWindow("Satellite Image", cv2.WINDOW_KEEPRATIO)
cv2.setMouseCallback("UAV Image", mouse_callback)

while True:
    # 在图像上绘制坐标（可选）
    uav_image = uav_data.uav_image[..., ::-1].copy()
    satellite_image = uav_data.satellite_image[..., ::-1].copy()
    cv2.putText(
        uav_image,
        f"Position: {uav_pos}",
        (size * 5, size * 15),
        cv2.FONT_HERSHEY_SIMPLEX,
        size * 0.3,
        (0, 255, 0),
        size,
    )
    cv2.putText(
        satellite_image,
        f"lonlat: {satellite_lonlat}",
        (size * 5, size * 15),
        cv2.FONT_HERSHEY_SIMPLEX,
        size * 0.3,
        (0, 255, 0),
        size,
    )
    cv2.circle(satellite_image, satellite_pos, size, (0, 255, 0), size)
    # 显示图像
    cv2.imshow("UAV Image", uav_image)
    cv2.imshow("Satellite Image", satellite_image)
    key = cv2.waitKey(1)
    # 按ESC键退出
    if key == 27:
        break
    elif key in [119, 97, 115, 100]:
        if key in [
            119,
        ]:  # w
            data_index -= 10
        elif key in [97]:  # a
            data_index -= 1
        elif key in [115]:  # s
            data_index += 10
        elif key in [100]:  # d
            data_index += 1
        uav_data = dataset[data_index]
        calibration = calibrate()
cv2.destroyAllWindows()